In [ ]:
import numpy as np 
import h5py
import matplotlib.pyplot as plt
import scipy.io
from utils import plot_jungfrau, combineRuns, get_tree, is_leaf, get_leaves, runNumToString, enable_underscore_cleanup
enable_underscore_cleanup() # Sets ipython hook to delete user defined varibles that start with _ at the end of each cell execution

In [ ]:
###############################################
#runNumbers = [15];
runNumbers = [15,16,17,18,19,20,21,22,23,24,25,26,27,28]
### Argon Runs ###
# runNumbers = [93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115]
folder = '/sdf/data/lcls/ds/cxi/cxil1037623/results/davidjr/'
# ###
# if runNumbers[0]==15:
#     folder = '/sdf/data/lcls/ds/cxi/cxil1037623/scratch/davidjr/dg2ipmReproc/';
# else:
#     folder = '/sdf/data/lcls/ds/cxi/cxil1037623/results/davidjr/'
###############################################
# (1) keys_to_combine: some keys loaded for each shot & stored per shot 
# (2) keys_to_sum: some keys loaded per each run and added 
# (3) keys_to_check : check if some keys exits and have same values in all runs and load these keys 
keys_to_combine = ['jungfrau4M/azav_mask0_azav', # Unfiltered
                   'jungfrau4M/azav_mask1_azav', # Filtered
                   'dg2ipmReproc/sum',
                   'dg2ipmReproc/peaks',
                   'dg2ipmReproc/xpos',
                   'dg2ipmReproc/ypos',
                   'gas_detector/f_11_ENRC',
                   'ebeam/photon_energy',
                   'evr/code_183',
                   'evr/code_137',
                   'evr/code_141',
                   'lightStatus/xray',
                  'jungfrau4M/Full_thres_sum',
                  'feeBld/hproj',
                  'lightStatus/laser',
                   'ipm_dg2/sum',
                   'unixTime',
                   'epicsUser/gasCell_pressure',
                  ]

keys_to_sum = ['Sums/jungfrau4M_calib']
#               'Sums/jungfrau4M_calib_thresADU1']

keys_to_check = ['UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_q',
                'UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_q',
                 'UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_idxq',
                 'UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_idxq',
                'UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_qbin',
                'UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_qbin',
                'UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_qbins',
                'UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_qbins',
                'UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_userMask',
                'UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_userMask',
                'UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_matrix_q', # This are only needed once
                'UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_matrix_phi', # This are only needed once
                'UserDataCfg/jungfrau4M/x',
                'UserDataCfg/jungfrau4M/y',
                'UserDataCfg/jungfrau4M/z',
                'UserDataCfg/jungfrau4M/cmask']
# Load the data in
_data = combineRuns(runNumbers, folder, keys_to_combine, keys_to_sum, keys_to_check, verbose=False)  # this is the function to load the data with defined keys

# Filtered Data
azavFiltered = np.squeeze(_data['jungfrau4M/azav_mask0_azav']) # I(q) : 1D azimuthal average of signals in each q bin
qbinFiltered = _data['UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_qbin'] # q bin-size
qFiltered = _data['UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_q'] # q bins 
qbinsFiltered = _data['UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_qbins'] # q bins
userMaskFiltered = _data['UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_userMask'].astype(bool) # User mask for this region
qbinSizeFiltered = np.bincount(_data['UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_idxq'])
# Unfiltered Data
azav = np.squeeze(_data['jungfrau4M/azav_mask1_azav']) # I(q) : 1D azimuthal average of signals in each q bin
qbin = _data['UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_qbin'] # q bin-size
q = _data['UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_q'] # q bins 
qbins = _data['UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_qbins'] # q bins
userMask = _data['UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_userMask'].astype(bool) # User mask for this region
qbinSize = np.bincount(_data['UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_idxq'])
# Other Data
matrix_q = _data['UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_matrix_q'].reshape(8,512,1024) # Q values J4M shaped
matrix_phi = _data['UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_matrix_phi'].reshape(8,512,1024) # phi valyes J4M shaped
xrayOn = _data['evr/code_137'].astype(bool)  # xray on events
xrayOn2 = _data['lightStatus/xray'].astype(bool)  # xray on events
laserOn = _data['lightStatus/laser'].astype(bool)  # xray on events
jungfrau_sum = _data['Sums/jungfrau4M_calib']  # Total Jungfrau detector counts summed in a run
#jungfrau_sum = data['Sums/jungfrau4M_calib_thresADU1']   # Total Jungfrau detector counts with Thresholds added, summed in a run 
x = _data['UserDataCfg/jungfrau4M/x'] # coordinates of Jungfrau detector x,y,z
y = _data['UserDataCfg/jungfrau4M/y']
z = _data['UserDataCfg/jungfrau4M/z'] 

cmask = _data['UserDataCfg/jungfrau4M/cmask'].astype(bool) # Mask for detector created 
run_indicator = _data['run_indicator'] # run indicator for each shot
#pressure = data['epicsAll/gasCell_pressure']  # pressure in gas cell
xray_energy = _data['gas_detector/f_11_ENRC']   # xray energy from gas detector (not calibrated to actual values)
xray_eV = _data['ebeam/photon_energy']    # x-ray energy energy in eV|
numPhotons = _data['jungfrau4M/Full_thres_sum']
spec = _data['feeBld/hproj'] # Shot to shot spectrometer
#dg2traces = data['CXI-DG2-BMMON-WF/ROI_area'] # Downsampling by a factor of 8, makes the refitting much better in the long run
gasPressure = _data['epicsUser/gasCell_pressure']
eventTime = _data['unixTime']
ipm = _data['dg2ipmReproc/sum']
ipmpeaks = _data['dg2ipmReproc/peaks']
xpos = _data['dg2ipmReproc/xpos']
ypos = _data['dg2ipmReproc/ypos']
#dg2tracesFull = data['CXI-DG2-BMMON-WF/ROI_rebin_data']

## Let's take a look at the statistics from the detectors: J4M, DG2-IPM, and XRT SPEC

In [ ]:
########## Different filter cutoffs
_J4M_cutoff = [0.1, 1];
_dg2_cutoff = [0.1, 0.95];
_spec_cutoff = [0.04, 0.19];
##########

# Precomputing the normalized values
scattered_xrays = np.nansum(azav,axis=-1)
scattered_xraysFiltered = np.nansum(azavFiltered,axis=-1)
_scattered_xrays_norm = scattered_xrays/scattered_xrays.max();
_ipm_norm = ipm/ipm.max();
_spec_norm = spec.sum(axis=1)/spec.sum(axis=1).max();


plt.figure(figsize=[17,5]) 
plt.subplot(1,3,1)
plt.hist(_scattered_xrays_norm,bins=200);
plt.axvline(_J4M_cutoff[0],color='r',linestyle='--')
plt.axvline(_J4M_cutoff[1],color='r',linestyle='--')
plt.yscale('log')
plt.title('J4M Sum Histogram')
plt.xlabel('Fraction of Maximum')
plt.ylabel('Counts')

plt.subplot(1,3,2)
plt.hist(_ipm_norm,bins=200);
plt.axvline(_dg2_cutoff[0],color='r',linestyle='--')
plt.axvline(_dg2_cutoff[1],color='r',linestyle='--')
plt.yscale('log')
plt.title('DG2 Sum Histogram')
plt.xlabel('Fraction of Maximum')
plt.ylabel('Counts')

plt.subplot(1,3,3)
plt.hist(_spec_norm,bins=200);
plt.axvline(_spec_cutoff[0],color='r',linestyle='--')
plt.axvline(_spec_cutoff[1],color='r',linestyle='--')
plt.yscale('log')
plt.title('XRT Sum Histogram')
plt.xlabel('Fraction of Maximum')
plt.ylabel('Counts')

plt.show()
goodIdx = np.logical_and.reduce([
    xrayOn2,
    _J4M_cutoff[0] <= _scattered_xrays_norm,
    _scattered_xrays_norm <= _J4M_cutoff[1],
    _dg2_cutoff[0] <= _ipm_norm,
    _ipm_norm <= _dg2_cutoff[1],
    _spec_cutoff[0] <= _spec_norm,
    _spec_norm <= _spec_cutoff[1],
    ~np.isnan(scattered_xrays)
])

azavFGood = azavFiltered[goodIdx]
azavGood = azav[goodIdx]
specGood = spec[goodIdx]
J4MSumFilter = scattered_xraysFiltered[goodIdx]
J4MSum = scattered_xrays[goodIdx]
gasPGood = gasPressure[goodIdx]
ipmGood = ipm[goodIdx]
# Displaying how much data was kept due to this filtering
_counts,_bins = np.histogram(goodIdx.astype(int),[0,1,2]);
print(f'Good data represents {_counts[1]/np.sum(_counts)*100:.2f}% of the total shots. ({_counts[1]} out of {np.sum(_counts)}).')

## Checking to see if the spectrometer output is offset by one shot or not.

In [ ]:
###### Set the offset Here ######
_offset = -1;
#################################
plt.figure(figsize=[17,5]) 
plt.subplot(1,3,1)
plt.hist2d(specGood.sum(axis=1)[0:-2],J4MSum[1:-1],bins=100);
plt.xlabel('XRT Spectrometer Sum')
plt.ylabel('J4M Sum')
plt.title('Spectrometer Offset: -1 Shot')
plt.subplot(1,3,2)
plt.hist2d(specGood.sum(axis=1),J4MSum,bins=100);
plt.xlabel('XRT Spectrometer Sum')
plt.ylabel('J4M Sum')
plt.title('Spectrometer Offset: 0 Shots')
plt.subplot(1,3,3)
plt.hist2d(specGood.sum(axis=1)[1:-1],J4MSum[0:-2],bins=100);
plt.xlabel('XRT Spectrometer Sum')
plt.ylabel('J4M Sum')
plt.title('Spectrometer Offset: +1 Shot')
# Determine slice indices based on offset
if _offset == -1:
    _spec_slice = slice(0, -2)
    _azav_slice = slice(1, -1)
elif _offset == 1:
    _spec_slice = slice(1, -1)
    azav_slice = slice(0, -2)
else:  # offset == 0 or any default
    _spec_slice = azav_slice = slice(None)

# Apply slices
specGood = spec[goodIdx][_spec_slice]
azavGood = azav[goodIdx][_azav_slice]
azavFGood = azavFiltered[goodIdx][_azav_slice]
J4MSum = scattered_xrays[goodIdx][_azav_slice]
J4MSumFilter = scattered_xraysFiltered[goodIdx][_azav_slice]
gasPGood = gasPressure[goodIdx][_azav_slice]
ipmGood = ipm[goodIdx][_azav_slice]
del spec, azav, azavFiltered, scattered_xrays, scattered_xraysFiltered, gasPressure, ipm

## Great, now we have isolated the good data and have fixed any offsets incurred from the XRT Spectromter. Now let's take a look at detector correlations.

### First, let's look at DG2-IPM In some more detail.

In [ ]:
plt.hist2d(ipmGood,J4MSum,bins=100);
plt.xlabel('DG2 IPM Sum');
plt.ylabel('J4M Sum');

### There is something going on here: If we look at the total signal from the J4M divided by the DG2-IPM reading for that shot we can compare it to the gas pressure in the cell.

In [ ]:
plt.figure(figsize=[12,5]);
# Plotting the I0 normalized J4M Sum
plt.subplot(1,2,1);
plt.plot(J4MSum/ipmGood);
plt.xlabel('Shot Number');
plt.ylabel('J4M Sum / DG2-IPM Sum');
# Plotting the gas pressure
plt.subplot(1,2,2);
plt.plot(gasPGood);
plt.xlabel('Shot Number');
plt.ylabel('IP Gas Pressure (torr)');

### They seem to match. Let's normalize the J4M sum to the gas pressure

In [ ]:
plt.figure(figsize=[12,5]);
plt.subplot(1,2,1);
plt.hist2d(ipmGood,J4MSum/gasPGood,bins=100);
plt.xlabel('DG2 IPM Sum');
plt.ylabel('J4M Sum / Gas Cell Pressure');
plt.subplot(1,2,2);
plt.hist2d(specGood.sum(axis=1),J4MSum/gasPGood,bins=100);
plt.xlabel('XRT Spectrometer Sum');
plt.ylabel('J4M Sum / Gas Cell Pressure');

### Much Better. Let's make some new variables which will store the good data

In [ ]:
azavNorm = azavGood/(gasPGood*ipmGood)[:, np.newaxis];
azavFNorm = azavFGood/(gasPGood*ipmGood)[:, np.newaxis];

### Now let's take a look at the spectral data and the J4M Data.

In [ ]:
# First, let's laod in the calibrated energy axis.
if runNumbers[0] == 15:
    energyAxis = np.load('ZnCalib/15-28.npy')

# Computing the stdev of each photon energy
_specMean = specGood.mean(axis=0);
_std = specGood.std(axis=0)/2;
plt.figure(figsize=[12,5]);
plt.plot(energyAxis,_specMean,label='Average',color='k');
plt.fill_between(energyAxis, _specMean - _std, _specMean + _std, alpha=0.3, label='±1/2 Std Dev')
plt.axvline(9.6586, color='r', linestyle='--', label='Zn K Edge') # Zinc K edge
for i in range(1,6):
    _shotIdx = np.random.randint(0, 1001)
    plt.plot(energyAxis,specGood[i*_shotIdx],linewidth=0.5,label=f'Shot {i*_shotIdx}');
plt.xlabel('Photon Energy (keV)');
plt.ylabel('Intesnity (arb.)');
plt.title('XRT Spectrometer Readings');
plt.axis('tight')
plt.legend();

### Before Taking the ghost imaging approach, let's try something simple first, since it may not be ghost imaging at all that we need

In [ ]:
ZnK = 9.6586 # Zn K edge in keV
_ZnIdx = int(np.round(np.searchsorted(energyAxis, ZnK))) # Find the index in the array closest to the Zn K edge
_specAbove = specGood[:,_ZnIdx:-1] # Split the spectrum at the Zn K edge and take the part that is above it.
_aboveSum = _specAbove.sum(axis=1) # Calculate the sum of each shot for the portion above the edge
_fracAbove = _aboveSum/(specGood.sum(axis=1)) # Find out what fraction of the pulses energy profile are above the edge

_aboveCutoff = 0.975 # Cutoff for pulses that are "Above the Zn K edge"
_belowCutoff = 0.2 # Cutoff for pulses that are "Below the Zn K edge"

plt.figure(figsize=[17,5]);
plt.subplot(1,3,1)
plt.hist(_fracAbove,bins=100); # Plotting a histogram of the fraction of the energy profile each pulse above the edge
plt.axvline(_aboveCutoff, color='r', linestyle='--',label='"Above" cutoff'); # Vertical line at the upper cuttoff
plt.axvline(_belowCutoff, color='r', linestyle='--',label='"Below" cutoff'); # Vertical line at the lower cutoff
plt.title('Fraction of the pulse energy \n profile above the Zn K edge Histogram')
plt.xlabel('Fraction of sum')
plt.ylabel('Counts')
plt.legend()

aboveIdx = _fracAbove>=_aboveCutoff # Creating indexing arrays based on the cutoff
belowIdx = _fracAbove<=_belowCutoff
plt.subplot(1,3,2)
plt.axvline(ZnK, color='r', linestyle='--', label='Zn K Edge') # Zinc K edge
for _i in range(0,5):
    plt.plot(energyAxis,specGood[aboveIdx][_i,:],label=f"Shot {_i}",linewidth=0.5)
plt.legend()
plt.title('First 5 shots "Above" the edge')
plt.ylabel('Intensity')
plt.xlabel('Photon Energy (keV)')

plt.subplot(1,3,3)
plt.axvline(ZnK, color='r', linestyle='--', label='Zn K Edge') # Zinc K edge
for _i in range(0,5):
    plt.plot(energyAxis,specGood[belowIdx][_i,:],label=f"Shot {_i}",linewidth=0.5)
plt.legend()
plt.title('First 5 shots "Below" the edge')
plt.ylabel('Intensity')
plt.xlabel('Photon Energy (keV)')
print(f'"Above" data represents {aboveIdx.sum()/len(_fracAbove)*100:.2f}% of the total good shots. ({aboveIdx.sum()} out of {len(_fracAbove)}).')
print(f'"Below" data represents {belowIdx.sum()/len(_fracAbove)*100:.2f}% of the total good shots. ({belowIdx.sum()} out of {len(_fracAbove)}).')

### Taking a look at how the indicies change sort out the combined correlation between the spectrometer sum and the J4M sum in the filtered region

In [ ]:
plt.figure(figsize=[17,5])
_xrange = [0, 2.5e9]
_yrange = [0, 370]
_hist_range = [_xrange, _yrange]
# Total filtered Signal
plt.subplot(1,3,1);
plt.hist2d(specGood.sum(axis=1), J4MSumFilter, bins=100, range=_hist_range);
plt.xlabel('XRT Spectrometer Sum');
plt.ylabel('J4M (Filtered) Sum');
plt.title('XRT vs J4M (Filtered) Sum Histogram');
# Shots that were "above" the edge
plt.subplot(1,3,2);
plt.hist2d(specGood[aboveIdx].sum(axis=1), J4MSumFilter[aboveIdx], bins=100, range=_hist_range);
plt.xlabel('XRT Spectrometer Sum');
plt.ylabel('J4M (Filtered) Sum');
plt.title('XRT vs J4M (Filtered) Sum Histogram "Above" edge');
# Shots that were "below" the edge
plt.subplot(1,3,3);
plt.hist2d(specGood[belowIdx].sum(axis=1), J4MSumFilter[belowIdx], bins=100, range=_hist_range);
plt.xlabel('XRT Spectrometer Sum');
plt.ylabel('J4M (Filtered) Sum');
plt.title('XRT vs J4M (Filtered) Sum Histogram "Below" edge');

### Also, we can plot the stuff we are excluding.

In [ ]:
_xrange = [0, 2.5e9]
_yrange = [0, 370]
_hist_range = [_xrange, _yrange]
plt.hist2d(specGood[~(belowIdx & aboveIdx)].sum(axis=1), J4MSumFilter[~(belowIdx & aboveIdx)], bins=100, range=_hist_range);
plt.xlabel('XRT Spectrometer Sum');
plt.ylabel('J4M (Filtered) Sum');
plt.title('XRT vs J4M (Filtered) Sum Histogram "In the middle"');

### Now let's take these new indicies and look at the azav data. We will compare each "Above" and "Below" section for the filtered and unfiltered data.

In [ ]:
# Finding the mean of the azavs in each slice of the data
_95AboveF = np.nanmean(azavFNorm[aboveIdx],axis=0);
_80BelowF = np.nanmean(azavFNorm[belowIdx],axis=0);
_95Above = np.nanmean(azavNorm[aboveIdx],axis=0);
_80Below = np.nanmean(azavNorm[belowIdx],axis=0);
_azavAvg = np.nanmean(azavNorm,axis=0);
# Now to normalize them to the value at the lowest q bin of the filtered data for comparing.
_95AboveFNorm = _95AboveF/np.nanmax(_95AboveF);
_80BelowFNorm = _80BelowF/np.nanmax(_80BelowF);
_95AboveNorm = _95Above/_95Above[np.nanargmax(_95AboveF)];
_80BelowNorm = _80Below/_80Below[np.nanargmax(_95AboveF)];
_azavAvgNorm = _azavAvg/_azavAvg[np.nanargmax(_95AboveF)];

plt.figure(figsize=[17,5])
plt.subplot(1,2,1)
plt.plot(q,_95AboveFNorm,label='Filtered, >97.5% Above Zn K ($F_A$)');
plt.plot(q,_80BelowFNorm,label='Filtered, >80% Below Zn K ($F_B$)');
plt.plot(q,_95AboveNorm,label='Unfiltered, >97.5% Above Zn K ($U_A$)');
plt.plot(q,_80BelowNorm,label='Unfiltered, >80% Below Zn K ($U_B$)');
plt.plot(q,_azavAvgNorm,label='Unfiltered ($U$)');
plt.legend();
plt.xlabel('q');
plt.ylabel('Azav normalized to value at lowest q of filtered data');
plt.title('Different Azavs');
plt.xlim(q[np.nanargmax(_95AboveF)],np.max(q));
plt.ylim([0,1]);
# And now to compare these further with a percent difference
plt.subplot(1,2,2)
plt.plot(q,100*(_95AboveFNorm-_azavAvgNorm)/_azavAvgNorm, label=r'$100\%\times{}\frac{F_A-U}{U}$');
plt.plot(q,100*(_80BelowFNorm-_azavAvgNorm)/_azavAvgNorm, label=r'$100\%\times{}\frac{F_B-U}{U}$');
plt.plot(q,100*(_95AboveNorm-_azavAvgNorm)/_azavAvgNorm, label=r'$100\%\times{}\frac{U_A-U}{U}$');
plt.plot(q,100*(_80BelowNorm-_azavAvgNorm)/_azavAvgNorm, label=r'$100\%\times{}\frac{U_B-U}{U}$');
plt.legend()
plt.xlabel('q')
plt.ylabel('Percent Difference (%)')
plt.title('Percent Difference between difference azavs')
plt.xlim(q[np.nanargmax(_95AboveF)],np.max(q));

### Let's crop the data into a specific range where we have the most statistics, then run the ghost imaging analysis

In [ ]:

# Subtract shot-averaged mean from each (center fluctuations)
spectrum_fluct = specGood - specGood.mean(axis=0, keepdims=True)   # shape (N, num_E)
scattering_fluct = azavNorm - azavNorm.mean(axis=0, keepdims=True)  # shape (N, num_q)

# Compute ghost image: correlation between spectrum fluctuations and scattering fluctuations
# shape will be (num_E, num_q)
ghost_image = np.einsum('ne,nq->eq', spectrum_fluct, scattering_fluct) / run_indicator.shape[0]

# Optional: Normalize each energy or q bin if needed
ghost_image /= np.std(spectrum_fluct, axis=0, keepdims=True).T * np.std(scattering_fluct, axis=0, keepdims=True)

In [ ]:
# Plot result
plt.imshow(ghost_image, aspect='auto', origin='lower',
           extent=[q.min(), q.max(), energyAxis.min(), energyAxis.max()], cmap='viridis')
plt.colorbar(label='Ghost signal')
plt.xlabel('q index')
plt.ylabel('Energy index')
plt.title('Spectrally-Resolved Ghost Image of Isotropic Scattering')
plt.clim(-0.1,0.3)
plt.show()